<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/00_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Landsat lake clarity: the full pipeline, Phases 1 to 6

One notebook, one runtime, top to bottom. Every table, figure, and model a phase
produces stays in memory and on Drive for the next phase, so there is no
cross-session file handoff to break. The client deliverable is built separately in
`07_deliver.ipynb`.

To run: set `EE_PROJECT` in the setup cell below only if you will run Phase 5
(Earth Engine), then Runtime then Run all. Phase 5 pauses once while the Earth
Engine export tasks finish; resume from the next cell when they are done.

- Phase 1: audit the national matchup record
- Phase 2: choose the target lakes and measure the ICC ceiling
- Phase 3: build the training set and interrogate the features
- Phase 4: train the regional model and attack it
- Phase 5: reconcile Collection 1 and 2, extract 2021-present from Earth Engine
- Phase 6: validate against Water Quality Portal field data


In [ ]:
# Setup: clone or update the repo, mount Drive, point the package at Drive data.
# The [gee] extra installs earthengine-api for Phase 5; Phases 1 to 4 ignore it.
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee]"], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

EE_PROJECT = "YOUR-EE-PROJECT"  # Phase 5 only: your registered noncommercial EE Cloud project

# Force a fresh import of the freshly pulled code. Colab keeps modules from a
# previous run in memory, so a git pull updates the files on disk while the
# kernel keeps running the old lakeclarity and silently ignores any fix. Purge
# them here so the import in the next cell always loads the pulled version.
for _stale in [m for m in list(sys.modules)
               if m == "lakeclarity" or m.startswith("lakeclarity.")]:
    del sys.modules[_stale]


# Phase 1 - Audit the published data

Fetches the LAGOS-US LANDSAT matchup table and LAGOS-US LOCUS lake metadata from EDI
onto Drive, converts them to Parquet once, and produces figures F1 to F5.

**Run this in Colab, not locally.** The EDI pull is ~550 MB and Colab sits in a Google
datacenter; a home connection takes hours for the same bytes.

Expectations were written down in `PLAN.md` before any of this ran. Two were already
wrong. That is the point of writing them down.

In [ ]:

# fetch once, convert once. Both steps skip if already on Drive.
import pandas as pd
from lakeclarity import audit, config, edi, features, locus, viz

viz.use_style()

edi.download("matchups")
edi.download("lake_information")
edi.download("lake_characteristics")

matchups = edi.load_matchups()
lakes = locus.load_lakes()

features.check_schema(matchups)
print(f"{len(matchups):,} matchups x {matchups.shape[1]} columns")
print(f"{len(lakes):,} lakes in LOCUS")
assert matchups.shape[1] == config.N_COLUMNS_PUBLISHED

In [ ]:
# the audit itself
avail = audit.secchi_availability(matchups)
print(avail)

comp = audit.completeness(matchups)
comp.to_csv(config.TABLE_DIR / "T01_completeness.csv")
print("\nconstant columns:", comp.index[comp.is_constant].tolist())

neg = audit.negative_reflectance_report(matchups)
neg.to_csv(config.TABLE_DIR / "T02_negative_reflectance.csv", index=False)
print("\n", neg.to_string(index=False))
print("\nany band negative: %.3f%% of Secchi matchups" % neg.attrs["any_band_negative_pct"])
print("mean Secchi where a band is negative : %.3f m" % neg.attrs["mean_secchi_any_negative"])
print("mean Secchi where all bands positive : %.3f m" % neg.attrs["mean_secchi_no_negative"])
print("\nIf the first is LARGER than the second, the standard quality filter is deleting")
print("the clear end of the distribution, and the whole lineage trained on a biased sample.")

for fn, fid, slug in [
    (audit.fig_missingness, "F01", "missingness"),
    (audit.fig_day_diff, "F02", "day_diff_window"),
    (audit.fig_coverage_by_satellite, "F03", "coverage_by_satellite"),
    (audit.fig_pixelcount, "F04", "pixelcount"),
    (audit.fig_secchi_transform, "F05", "secchi_transform"),
]:
    print("wrote", viz.save(fn(matchups), fid, slug).name)

# Phase 2 - Choose the target lakes, and measure the ceiling

Region: **Northern Lower Michigan** (clear glacial kettle lakes, same optical regime as the
New Hampshire reference lakes, with decades of Cooperative Lakes Monitoring Program Secchi).

**The gate.** One large lake and one small one, both with enough FIELD July-years to support a
client-style July-annual-mean validation. Selection is on distinct years with a July field
Secchi reading in the Water Quality Portal, NOT on coincident satellite/in-situ matchups.
Matchups require a clean pass within three days of a reading and badly undercount the
achievable validation sample; the client's own r = -0.22 for Squam used the full field record,
not matchups. If no small lake qualifies, `select_target_lakes` raises.

**The ceiling.** The intraclass correlation of log Secchi across the region's lakes: the
fraction of the pooled signal that says nothing about tracking one lake through time. It is the
number that reconciles a published R-squared of 0.637 with a per-lake r of -0.22.

In [ ]:
# training set (all Michigan) and target region (northern counties)
import pandas as pd
from lakeclarity import config, eda, edi, features, locus, selection, viz, wqp

viz.use_style()

matchups = edi.load_matchups()
lakes = locus.load_lakes()

# Piper, Glines & Rose trained on ALL Wisconsin lakes in AquaSat, not just their
# 127 study lakes. The equivalent is every Michigan lake. Target lakes come from
# the narrower northern-county region and are held out of training entirely.
train_lk = locus.train_lakes(lakes)
region_lk = locus.region_lakes(lakes)
print(f"{len(train_lk):,} {config.TRAIN_REGION_NAME} lakes, "
      f"{len(region_lk):,} {config.REGION_NAME} lakes in LOCUS")

region = matchups[matchups["lagoslakeid"].isin(train_lk["lagoslakeid"])].copy()
region = region[region[config.TARGET].notna()]
region = features.add_time_columns(region)
region = features.add_log_target(region)
print(f"{len(region):,} Secchi matchups on {region.lagoslakeid.nunique():,} lakes (training supply)")
region.to_parquet(config.INTERIM_DIR / "region_matchups.parquet", index=False)

In [ ]:
# FIELD coverage from the Water Quality Portal (the selection metric)
# Confirm the characteristic-name choice, then pull field Secchi + station coords,
# map each station to its nearest lake, and count distinct July-years per lake.
print(wqp.characteristic_coverage().to_string(index=False), "\n")

field = wqp.fetch_secchi()                 # cached to Drive after first pull
stations = wqp.fetch_stations()            # site coordinates
site_to_lake = wqp.map_sites_to_lakes(stations, lakes)
field_cov = wqp.lake_field_coverage(field, site_to_lake)

site_to_lake.to_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet", index=False)
field_cov.to_csv(config.TABLE_DIR / "T04b_field_coverage.csv")
print(f"{len(field):,} field Secchi records, {len(stations):,} stations, "
      f"{len(site_to_lake):,} mapped to a lake")
print(f"lakes with >= {config.MIN_FIELD_JULY_YEARS} field July-years: "
      f"{(field_cov['field_july_years'] >= config.MIN_FIELD_JULY_YEARS).sum()}")
print(field_cov.head(12).to_string())

In [ ]:
# THE CEILING LADDER. Run this before choosing anything.
# ICC is a property of the population, not of nature. Tighten the population from
# the whole US to one optically homogeneous region and the between-lake variance
# that a pooled R2 mostly measures collapses, exposing the temporal signal a
# regional model can actually learn. Compute it at three tiers to show the mechanism.
national = matchups[matchups[config.TARGET].notna()].copy()
national = features.add_log_target(national)

# The homogeneous target region (northern counties), NOT the whole training state.
# This is where between-lake variance is expected to collapse.
homreg = matchups[matchups["lagoslakeid"].isin(region_lk["lagoslakeid"])].copy()
homreg = homreg[homreg[config.TARGET].notna()]
homreg = features.add_log_target(homreg)

tiers = {
    "national (all US)": selection.ceiling_report(national),
    f"{config.TRAIN_REGION_NAME} (state)": selection.ceiling_report(region),
    config.REGION_NAME: selection.ceiling_report(homreg),
}
labels = list(tiers)

print(f"{'':>24}" + "".join(f"{l:>24}" for l in labels))
for k in ["icc", "between_lake_sd_log10", "within_lake_sd_log10", "n_lakes", "n_obs"]:
    print(f"{k:>24}" + "".join(f"{tiers[l][k]:>24,.3f}" for l in labels))

nat, hom = tiers[labels[0]], tiers[labels[-1]]
print(f"\nNational: {nat['pct_variance_between_lakes']:.0f}% of the variance is lakes")
print("differing from one another. That is what a pooled R2 of 0.637 mostly measures,")
print("and it says nothing about tracking one lake through time.")
print(f"\nInside {config.REGION_NAME} the lakes resemble each other, so between-lake")
print(f"variance falls to {hom['pct_variance_between_lakes']:.0f}% and "
      f"{hom['pct_variance_within_lakes']:.0f}% of what remains is genuine")
print("temporal signal, the signal a regional model learns and a national model drowns.")

pd.DataFrame(tiers).to_csv(config.TABLE_DIR / "T03b_icc_ladder.csv")
pd.Series(tiers[config.REGION_NAME]).to_csv(config.TABLE_DIR / "T03_variance_ceiling.csv")

In [ ]:
# the gate
candidates = selection.candidate_table(region, region_lk, field_cov)
candidates.to_csv(config.TABLE_DIR / "T04_candidate_lakes.csv", index=False)
print(candidates.head(15)[["lagoslakeid", "lake_name", "lake_waterarea_ha",
                            "field_july_years", "field_year_first", "field_year_last",
                            "field_secchi_mean", "n_matchups"]].to_string(index=False))

try:
    targets = selection.select_target_lakes(candidates)
    print("\n" + targets.summary())
    pd.Series({"large": targets.ids[0], "small": targets.ids[1]}).to_csv(
        config.TABLE_DIR / "T05_target_lakes.csv")
except selection.NoEligibleLakeError as exc:
    print("\nGATE FAILED:", exc)
    print("\nDo not work around this. Either relax MIN_FIELD_JULY_YEARS with a written")
    print("justification, widen the region, or accept that the small-lake case cannot")
    print("be validated with the data that exists.")
    raise

In [ ]:
# figures F6 to F9
# F08 contrasts the two extremes of the ladder: the national distribution against
# the homogeneous target region, where between-lake variance collapses.
for fig, fid, slug in [
    (selection.fig_region_map(candidates, targets), "F06", "region_map"),
    (selection.fig_area_vs_pixels(candidates, targets), "F07", "area_vs_pixels"),
    (selection.fig_variance_decomposition(homreg, national=national,
                                          train_label=config.REGION_NAME),
     "F08", "variance_decomposition"),
    (selection.fig_candidate_timeseries(field, site_to_lake, candidates), "F09", "candidate_timeseries"),
]:
    print("wrote", viz.save(fig, fid, slug).name)

# Phase 3 - Build the training set, interrogate the features

Three questions, each with a figure that can embarrass us.

1. **Is the quality filter neutral?** Negative median reflectance is a Collection 1
   aerosol over-correction. If it happens preferentially over dark water, dropping those
   rows drops the clear end of the Secchi distribution, and every model in this lineage
   trained on a biased sample. F10 reports the mean Secchi on both sides of every filter
   step.
2. **Are the 15 band ratios independent?** They are algebraic functions of 6 medians.
   F11 gives the condition number. This is why Phase 4 reports permutation importance
   rather than Gini.
3. **Is a forty-year clarity trend real, or is it the satellites?** F13 plots band
   reflectance for a large, clear, stable lake. It should be flat. If it steps at 2013,
   any uncorrected trend is an instrument artifact.

In [ ]:
# the filter chain, with the target lakes removed first
import pandas as pd
from lakeclarity import config, explore, features, locus, viz

viz.use_style()

region = pd.read_parquet(config.INTERIM_DIR / "region_matchups.parquet")
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
holdout = [int(targets["large"]), int(targets["small"])]
print("holding out lakes:", holdout)

train, filter_log = features.build_training_frame(region, holdout_lake_ids=holdout)
waterfall = filter_log.to_frame()
waterfall.to_csv(config.TABLE_DIR / "T06_filter_waterfall.csv", index=False)
print(waterfall.to_string(index=False))

train = locus.attach_lake_metadata(train)
train.to_parquet(config.INTERIM_DIR / "train.parquet", index=False)
print(f"\ntraining set: {len(train):,} matchups on {train.lagoslakeid.nunique():,} lakes")
assert not set(holdout) & set(train["lagoslakeid"]), "target lake leaked into training"

In [ ]:
# does the QA filter delete clear water?
shift = waterfall.loc[waterfall["step"] == "negative_reflectance", "secchi_shift_m"]
if len(shift) and shift.iloc[0] < -0.02:
    print(f"CONFIRMED: dropping negative-reflectance rows lowers mean Secchi by "
          f"{abs(shift.iloc[0]):.3f} m.")
    print("The standard quality filter is biased against clear water, which is the water")
    print("this model is asked to predict. Report this.")
elif len(shift) and shift.iloc[0] > 0.02:
    print(f"REFUTED: the filter *raises* mean Secchi by {shift.iloc[0]:.3f} m.")
    print("Negative reflectance is concentrated in turbid water, not clear. The Phase 1")
    print("hypothesis was wrong; say so in the report rather than quietly dropping it.")
else:
    print("NEUTRAL: the filter does not measurably shift the Secchi distribution.")

In [ ]:
# collinearity and univariate signal
X = features.feature_matrix(train)
y = train[config.LOG_TARGET]

cond = explore.predictor_condition_number(X)
print(f"condition number of the standardised predictor matrix: {cond:,.0f}")
print("Gini importance is not interpretable on a matrix this collinear. Phase 4 uses")
print("permutation importance and shows both side by side.")

mi = explore.mutual_information(X, y)
mi.to_csv(config.TABLE_DIR / "T07_mutual_information.csv")
print("\ntop 8 predictors by mutual information with log10 Secchi:")
print(mi.head(8).to_string())

top_is_blue = any(c in config.CDOM_RATIOS for c in mi.head(3).index)
print(f"\nblue-family ratio in the top three: {top_is_blue}")
print("CDOM absorbs in the blue and barely scatters, so blue-family ratios should lead in")
print("stained glacial lakes. If red and NIR lead instead, this region is sediment-driven")
print("and the analogy to New Hampshire is weaker than assumed. That is worth knowing now.")

In [ ]:
# figures F10 to F14
ref_lake = explore.pick_stable_reference_lake(train)
print("drift reference lake:", train.loc[train.lagoslakeid == ref_lake, "lake_name"].iloc[0])

for fig, fid, slug in [
    (explore.fig_filter_waterfall(filter_log), "F10", "filter_waterfall"),
    (explore.fig_correlation_heatmap(X), "F11", "predictor_collinearity"),
    (explore.fig_mutual_information(X, y), "F12", "mutual_information"),
    (explore.fig_stable_lake_drift(train, ref_lake), "F13", "stable_lake_drift"),
    (explore.fig_seasonality(train), "F14", "seasonality"),
]:
    print("wrote", viz.save(fig, fid, slug).name)

# Phase 4 - Train the regional model, then attack it

Every split is a `GroupKFold` on `lagoslakeid`. A lake appears in exactly one test fold
and nothing about it is visible to the model that predicts it. With most of the variance
sitting between lakes, a random split would let the forest recognise a lake it has
already seen and report a score it cannot reproduce anywhere.

The output that matters is **F20**: the distribution of within-lake correlation across
held-out lakes, for the national model and the regional one, with the client's r = -0.22
drawn on it.

In [ ]:
# cross-validate with lakes held out whole
import joblib
import pandas as pd
from lakeclarity import config, features, model, viz

viz.use_style()
train = pd.read_parquet(config.INTERIM_DIR / "train.parquet")

cv = model.grouped_cv(train, n_splits=5)
metrics = cv.pooled_metrics()
for k, v in metrics.items():
    print(f"{k:>18}: {v:,.4f}" if isinstance(v, float) else f"{k:>18}: {v:,}")

cv.fold_scores.to_csv(config.TABLE_DIR / "T08_fold_scores.csv", index=False)
cv.frame.to_parquet(config.INTERIM_DIR / "oof_predictions.parquet", index=False)
print("\nfold-to-fold R2 spread:", cv.fold_scores["r2_log"].round(3).tolist())

In [ ]:
# THE HEADLINE. Pooled skill is not per-lake skill.
per_lake = model.per_lake_skill(cv, min_obs=8)
per_lake.to_csv(config.TABLE_DIR / "T09_per_lake_skill.csv", index=False)

print(f"pooled Pearson r across all held-out matchups : {metrics['pearson_r_pooled']:+.3f}")
print(f"median WITHIN-lake r across held-out lakes     : {per_lake['r'].median():+.3f}")
print(f"fraction of held-out lakes with negative r     : {(per_lake['r'] < 0).mean():.0%}")
print(f"lakes evaluated                                : {len(per_lake)}")
print()
print("If the second number is far below the first, the pooled score is carried by")
print("between-lake variance and says nothing about tracking one lake through time.")
print("That is the gap the client fell into, and r = -0.22 on Squam is a sample from it.")

In [ ]:
# the national model, evaluated on identical rows
# LAGOS_US_LANDSAT_Predictions_v1_QAQC is 7.5 GB. Stream it, keep our lakes only.
from lakeclarity import edi

national_path = config.INTERIM_DIR / "national_predictions_region.parquet"
if not national_path.exists():
    src = edi.download("predictions")
    edi.stream_filter(src, national_path, keep_ids=train["lagoslakeid"].unique())

national = pd.read_parquet(national_path)
national["SENSING_TIME"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)

# Join the national prediction onto the same observations our model was scored on.
obs = cv.frame.copy()
obs["key"] = obs["lagoslakeid"].astype(str) + "_" + obs["year"].astype(str) + "_" + obs["doy"].astype(str)
national["year"] = national["SENSING_TIME"].dt.year
national["doy"] = national["SENSING_TIME"].dt.dayofyear
national["key"] = national["lagoslakeid"].astype(str) + "_" + national["year"].astype(str) + "_" + national["doy"].astype(str)

nat = obs.merge(national[["key", "Secchi_predicted", "QAQC_recommend"]], on="key", how="inner")
nat = nat[nat["QAQC_recommend"].astype(str).str.upper() == "TRUE"]
nat = nat.rename(columns={"Secchi_predicted": "y_pred_m"})
print(f"{len(nat):,} observations have both a regional and a national prediction")

comparison = model.compare_to_national(cv.frame, nat)
comparison.to_csv(config.TABLE_DIR / "T10_per_lake_comparison.csv", index=False)
print(comparison.groupby("model")["r"].describe().round(3).to_string())

In [ ]:
# importance, on a model that has seen every training lake
X = features.feature_matrix(train)
y = train[config.LOG_TARGET]

rf_full = model.fit_random_forest(X, y)
joblib.dump(rf_full, config.PROCESSED_DIR / "regional_rf.joblib")

imp = model.importance_comparison(rf_full, X, y, n_repeats=10)
imp.to_csv(config.TABLE_DIR / "T11_importance.csv", index=False)
print(imp.head(12).to_string(index=False))
print("\nGini and permutation will disagree. The 15 ratios are algebraic functions of 6")
print("medians, and Gini importance is biased toward correlated predictors. Trust the")
print("permutation column, and show both.")

In [ ]:
# figures F15 to F20
top_feature = imp.iloc[0]["feature"]

for fig, fid, slug in [
    (model.fig_fold_scores(cv), "F15", "fold_scores"),
    (model.fig_observed_vs_predicted(cv), "F16", "observed_vs_predicted"),
    (model.fig_residual_grid(cv), "F17", "residual_grid"),
    (model.fig_importance(imp), "F18", "importance"),
    (model.fig_partial_dependence(rf_full, X, top_feature), "F19", "partial_dependence"),
    (model.fig_per_lake_skill(comparison), "F20", "per_lake_skill"),
]:
    print("wrote", viz.save(fig, fid, slug).name)

# Phase 5 - Predict 1984-present, and prove the sensors agree

The prediction is one line. The work is proving the 1984-present series measures lakes,
not satellites.

1. Predict the target lakes from `compiledRS` (Collection 1, 1984-2020).
2. Extract 2021-present from Earth Engine (Collection 2), with the ratios and KIVU
   computed **per pixel then reduced**, because the published ratios are medians of
   pixel-wise ratios, not ratios of medians (established in Phase 5's tests).
3. Fit the Collection 1 to Collection 2 handoff on the 2013-2020 overlap, apply it,
   and report how many centimetres of apparent clarity the collection change invents.

In [ ]:
# predict the Collection 1 record from compiledRS
import joblib
import numpy as np
import pandas as pd
from lakeclarity import config, edi, features, gee, harmonize, locus, predict, viz

viz.use_style()

rf = joblib.load(config.PROCESSED_DIR / "regional_rf.joblib")
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]

# compiledRS is 10.1 GB; stream it and keep only the two target lakes.
c1_path = config.INTERIM_DIR / "compiledRS_targets.parquet"
if not c1_path.exists():
    src = edi.download("compiled_rs")
    edi.stream_filter(src, c1_path, keep_ids=target_ids)

c1 = pd.read_parquet(c1_path)
c1 = features.add_time_columns(c1)
c1 = c1[c1["Pixelcount"] >= config.MIN_PIXELCOUNT]
c1 = c1[~features.has_negative_reflectance(c1)]
c1["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c1))
print(f"Collection 1: {len(c1):,} usable passes, {c1.year.min()}-{c1.year.max()}")

In [ ]:
# extract Collection 2 (2013-present) from Earth Engine
# 2013-2020 overlaps compiledRS and is used to fit the handoff; 2021+ is the extension.
gee.initialize(EE_PROJECT)
lakes = locus.load_lakes().set_index("lagoslakeid")

for lake_id in target_ids:
    row = lakes.loc[lake_id]
    geom = gee.lake_geometry(lake_id, row["lake_lat_decdeg"], row["lake_lon_decdeg"])
    fc = gee.extract_lake_timeseries(geom, lake_id, start="2013-01-01", end="2027-01-01")
    task = gee.export_to_drive(fc, description=f"c2_lake_{lake_id}")
    print(f"export started for lake {lake_id}: task {task.id}")

print("\nWait for the Earth Engine tasks to finish (Tasks tab or code.earthengine.google.com),")
print("then re-run from the next cell. The CSVs land in Drive/lake-clarity.")

In [ ]:
# load the Earth Engine exports, put them in published column names
c2_parts = []
for lake_id in target_ids:
    df = pd.read_csv(config.RAW_DIR.parent / f"c2_lake_{lake_id}.csv")
    df = gee.rename_gee_columns(df)
    df["lagoslakeid"] = lake_id
    c2_parts.append(df)
c2 = pd.concat(c2_parts, ignore_index=True)
c2 = features.add_time_columns(c2)
print(f"Collection 2: {len(c2):,} passes, {c2.year.min()}-{c2.year.max()}")
print("columns present:", sorted(set(config.FEATURES) & set(c2.columns)) == sorted(config.FEATURES))

In [ ]:
# fit and apply the Collection 1 -> Collection 2 handoff
# Match C1 and C2 passes of the same lake within one day, 2013-2020.
def overlap_frame(c1, c2, ids, tol_days=1):
    parts = []
    for lake_id in ids:
        a = c1[(c1.lagoslakeid == lake_id) & (c1.year.between(2013, 2020))].copy()
        b = c2[(c2.lagoslakeid == lake_id) & (c2.year.between(2013, 2020))].copy()
        a = a.sort_values("sensing_dt"); b = b.sort_values("sensing_dt")
        m = pd.merge_asof(a, b, on="sensing_dt", tolerance=pd.Timedelta(days=tol_days),
                          direction="nearest", suffixes=("_c1", "_c2"))
        parts.append(m.dropna(subset=[f"{config.BANDS[0]}median_c2"]))
    return pd.concat(parts, ignore_index=True)

modelled = [c for c in config.FEATURES]
overlap = overlap_frame(c1, c2, target_ids)
coefs = harmonize.fit_handoff(overlap, columns=modelled)
coefs.to_csv(config.TABLE_DIR / "T12_handoff_coefficients.csv")
print(coefs[["slope", "intercept", "r2", "n"]].round(4).to_string())

c2_2021 = c2[c2.year >= 2021].copy()
c2_corrected = harmonize.apply_handoff(c2_2021, coefs)
c2_corrected["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c2_corrected))
c2_2021["secchi_predicted_m"] = 10 ** rf.predict(features.feature_matrix(c2_2021))  # uncorrected

In [ ]:
# how many centimetres does the collection change invent?
shift = harmonize.secchi_shift_from_handoff(rf, c2_2021, c2_corrected)
ceiling = pd.read_csv(config.TABLE_DIR / "T03_variance_ceiling.csv", index_col=0).squeeze()
within_sd_m = float(10 ** ceiling["within_lake_sd_log10"] - 1) * c1["secchi_predicted_m"].median()

print(f"median absolute Secchi shift from the handoff : {shift['abs_median_shift_cm']:.1f} cm")
print(f"95th percentile                               : {shift['p95_abs_shift_cm']:.1f} cm")
print(f"a lake's real interannual 1 sigma             : {100*within_sd_m:.1f} cm")
print("\nIf the shift is a large fraction of the interannual signal, the correction is")
print("not optional. This is the argument for insisting on Earth Engine experience.")

corrected_full = pd.concat([c1, c2_corrected], ignore_index=True)
uncorrected_full = pd.concat([c1, c2_2021], ignore_index=True)
corrected_full.to_parquet(config.PROCESSED_DIR / "predictions_full.parquet", index=False)

In [ ]:
# figures F21 to F24
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    passes = corrected_full[corrected_full.lagoslakeid == lake_id]
    print("wrote", viz.save(predict.fig_pass_availability(passes, name), "F21", f"passes_{lake_id}").name)
    print("wrote", viz.save(predict.fig_full_timeseries(
        passes, name, uncorrected=uncorrected_full[uncorrected_full.lagoslakeid == lake_id]),
        "F24", f"timeseries_{lake_id}").name)

print("wrote", viz.save(predict.fig_collection_agreement(coefs, overlap), "F22", "collection_agreement").name)
print("wrote", viz.save(predict.fig_secchi_shift(shift, within_sd_m), "F23", "secchi_shift").name)

# Phase 6 - Validate the way the client validates, then stress it

Reproduce the client's diagnostic (Pearson r between July annual means, n = years) for
both models on the same lakes and field data, then add the three things the client did
not: a bootstrap CI on r, a sensitivity grid over the analyst's discretionary choices,
and a provenance check of Water Quality Portal Secchi against LAGOS-US LIMNO.

The credibility anchor: the national model should fail on our Michigan lakes too.
Reproducing the client's failure on different lakes in a different state, before claiming
to fix it, proves the failure is structural rather than specific to Squam.

In [ ]:
# pull field Secchi from the Water Quality Portal REST API
import pandas as pd
from lakeclarity import config, locus, validate, viz, wqp

viz.use_style()
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]
lakes = locus.load_lakes().set_index("lagoslakeid")

# Confirm the characteristic-name choice before relying on it.
coverage = wqp.characteristic_coverage()
print(coverage.to_string(index=False))

field = wqp.fetch_secchi()  # cached to Drive after the first pull
# Attach lagoslakeid to every field reading via the Phase 2 site->lake map, so
# validation can filter field Secchi to a specific target lake.
site_to_lake = pd.read_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet")
field = field.merge(site_to_lake[["site_id", "lagoslakeid"]], on="site_id", how="left")
print(f"\n{len(field):,} Secchi records in the region, {field.year.min()}-{field.year.max()}")
print("mapped to a target lake:", field["lagoslakeid"].isin(target_ids).sum(), "records")

In [ ]:
# the like-for-like comparison, with bootstrap intervals
predictions = pd.read_parquet(config.PROCESSED_DIR / "predictions_full.parquet")
national = pd.read_parquet(config.INTERIM_DIR / "national_predictions_region.parquet")
national = national.rename(columns={"Secchi_predicted": "secchi_predicted_m"})
national["sensing_dt"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)
national["year"] = national["sensing_dt"].dt.year
national["month"] = national["sensing_dt"].dt.month

# Field Secchi per target lake, via the site->lake map attached in cell 2. If the
# client supplies their own station list, filter `field` to it here instead.
results = []
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    obs = field[field["lagoslakeid"] == lake_id]
    for label, pred in [("regional", predictions), ("national", national)]:
        res = validate.july_validation(pred, obs, lake_id, name, label)
        if res:
            results.append(res)
            print(res.sentence())

In [ ]:
# does the result survive the analyst's choices?
# Rebuild predictions under each configuration, then grid the headline r.
from itertools import product

configs = {}
for window, floor, qa, months in product([1, 3], [10, 25], ["on"], [(7,), (6, 7, 8)]):
    p = predictions[predictions["Pixelcount"] >= floor]
    configs[(window, floor, qa, months)] = p

for lake_id in target_ids:
    grid = validate.sensitivity_grid(configs, field, lake_id)
    grid.to_csv(config.TABLE_DIR / f"T13_sensitivity_{lake_id}.csv", index=False)
    print(f"\nlake {lake_id} ({lakes.loc[lake_id, 'lake_name']}):")
    print(grid.to_string(index=False))
    if grid["r"].notna().any():
        print(f"  r ranges [{grid['r'].min():+.2f}, {grid['r'].max():+.2f}] across the grid")

In [ ]:
# figures F25 to F28
matchups = pd.read_parquet(config.INTERIM_DIR / "matchups.parquet") if (config.INTERIM_DIR / "matchups.parquet").exists() else None

for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    print("wrote", viz.save(validate.fig_validation_overlay(
        field, predictions, national, lake_id, name), "F25", f"overlay_{lake_id}").name)
    grid = validate.sensitivity_grid(configs, field, lake_id)
    print("wrote", viz.save(validate.fig_sensitivity_grid(grid, name), "F27", f"sensitivity_{lake_id}").name)
    if matchups is not None:
        merged = validate.provenance_check(field, matchups, lake_id)
        if len(merged) > 2:
            print("wrote", viz.save(validate.fig_provenance(merged, name), "F28", f"provenance_{lake_id}").name)

print("wrote", viz.save(validate.fig_bootstrap_r(results), "F26", "bootstrap_r").name)